In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Annotated, Dict, List, Literal
import re
import json

import pandas as pd
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.types import Command

# Optional LLM: Gemini
#pip install langchain-google-genai
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "xxxxx"  # Replace with your actual API key

In [5]:
# ============================================================
# 1. MODEL
# ============================================================
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0,
    google_api_key=os.environ["GOOGLE_API_KEY"],
)

In [6]:
# ============================================================
# 2. STATE
# ============================================================

class DataQualityState(MessagesState):
    dataset_path: str
    current_dataset_path: str
    final_dataset_path: str
    team_reports: List[dict]
    issue_registry: List[dict]
    reliability_score: float
    final_report: str

In [7]:
# ============================================================
# 3. ARTIFACTS DIR
# ============================================================

ARTIFACT_DIR = Path("./artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
# ============================================================
# 4. BASIC HELPERS
# ============================================================

def load_df(path: str) -> pd.DataFrame:
    return pd.read_csv(path)


def save_df(df: pd.DataFrame, path: str) -> str:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    return path


def build_team_report(
    team_name: str,
    input_dataset: str,
    output_dataset: str,
    tasks_executed: List[str],
    issues_found: List[str],
    changes_applied: List[str],
    changes_not_applied: List[str],
    handoff_summary: str,
) -> dict:
    return {
        "team_name": team_name,
        "input_dataset": input_dataset,
        "output_dataset": output_dataset,
        "tasks_executed": tasks_executed,
        "issues_found": issues_found,
        "changes_applied": changes_applied,
        "changes_not_applied": changes_not_applied,
        "handoff_summary": handoff_summary,
    }


def append_report(state: DataQualityState, report: dict) -> List[dict]:
    reports = list(state.get("team_reports", []))
    reports.append(report)
    return reports


def append_issues(state: DataQualityState, issues: List[dict]) -> List[dict]:
    registry = list(state.get("issue_registry", []))
    registry.extend(issues)
    return registry


def safe_snake_case(name: str) -> str:
    cleaned = re.sub(r"[^a-zA-Z0-9]+", "_", str(name).strip())
    cleaned = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", cleaned)
    cleaned = cleaned.lower().strip("_")
    if cleaned and cleaned[0].isdigit():
        cleaned = f"col_{cleaned}"
    return cleaned or "unnamed_column"


def standardize_missing_markers(df: pd.DataFrame) -> pd.DataFrame:
    placeholders = {"", "N/A", "n/a", "-", "unknown", "UNKNOWN", "null", "NULL"}
    for col in df.columns:
        df[col] = df[col].replace(list(placeholders), pd.NA)
    return df


def save_json(obj: dict | list, path: str) -> str:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    return path


In [9]:
# ============================================================
# 5. TOOLS
# ============================================================

@tool
def dataset_metadata_tool(dataset_path: Annotated[str, "Path to dataset"]) -> dict:
    """Return basic dataset metadata."""
    df = load_df(dataset_path)
    return {
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1]),
        "column_names": list(df.columns),
        "sample_records": df.head(5).to_dict(orient="records"),
    }


@tool
def copy_dataset_tool(
    dataset_path: Annotated[str, "Original dataset path"],
    output_path: Annotated[str, "Destination path"],
) -> str:
    """Create a working copy of the dataset."""
    df = load_df(dataset_path)
    return save_df(df, output_path)


@tool
def write_team_report_tool(
    report: Annotated[dict, "Team report"],
    output_path: Annotated[str, "Path to save report json"],
) -> str:
    """Save a team report to json."""
    return save_json(report, output_path)


@tool
def write_final_report_tool(
    final_report: Annotated[str, "Full final report text"],
    output_path: Annotated[str, "Path to save final report"],
) -> str:
    """Save final report to a text file."""
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(final_report)
    return output_path

In [10]:
# ============================================================
# 6. TEAM IMPLEMENTATIONS
#    Names exactly as in the PDF:
#    Schema Validation
#    Completeness Analysis
#    Consistency Validation
#    Anomaly Detection
#    Remediation
# ============================================================

def schema_validation_team(state: DataQualityState) -> Command[Literal["supervisor"]]:
    input_path = state["current_dataset_path"]
    df = load_df(input_path)

    tasks_executed = [
        "Checked column naming conventions",
        "Standardized invalid column names",
        "Applied safe numeric coercion on clearly numeric columns",
    ]
    issues_found = []
    changes_applied = []
    changes_not_applied = []
    issues = []

    # 1. rename columns
    original_columns = list(df.columns)
    rename_map = {}
    seen = set()

    for col in original_columns:
        new_name = safe_snake_case(col)
        candidate = new_name
        suffix = 1
        while candidate in seen:
            suffix += 1
            candidate = f"{new_name}_{suffix}"
        seen.add(candidate)

        if candidate != col:
            rename_map[col] = candidate
            issues_found.append(f"Invalid or inconsistent column name: {col}")
            issues.append(
                {
                    "team": "Schema Validation",
                    "issue": f"Column renamed from '{col}' to '{candidate}'",
                    "severity": "medium",
                }
            )

    if rename_map:
        df = df.rename(columns=rename_map)
        changes_applied.append(f"Renamed {len(rename_map)} columns to snake_case")

    # 2. safe numeric coercion if strongly numeric-like
    for col in df.columns:
        series = df[col].dropna().astype(str).head(50)
        if len(series) == 0:
            continue
        numeric_like = 0
        for value in series:
            try:
                float(value.replace(",", "."))
                numeric_like += 1
            except Exception:
                pass

        if numeric_like >= max(3, int(len(series) * 0.8)):
            old_non_null = int(df[col].notna().sum())
            coerced = pd.to_numeric(df[col], errors="coerce")
            new_non_null = int(coerced.notna().sum())
            # apply only if we don't destroy too much information
            if new_non_null >= int(old_non_null * 0.8):
                df[col] = coerced
                changes_applied.append(f"Coerced column to numeric: {col}")

    output_path = str(ARTIFACT_DIR / "schema_validation_output.csv")
    save_df(df, output_path)

    report = build_team_report(
        team_name="Schema Validation",
        input_dataset=input_path,
        output_dataset=output_path,
        tasks_executed=tasks_executed,
        issues_found=issues_found if issues_found else ["No schema issues found"],
        changes_applied=changes_applied if changes_applied else ["No changes applied"],
        changes_not_applied=changes_not_applied if changes_not_applied else ["No deferred schema actions"],
        handoff_summary="Schema standardized. Cleaned dataset passed to Completeness Analysis."
    )

    write_team_report_tool.invoke(
        {
            "report": report,
            "output_path": str(ARTIFACT_DIR / "schema_validation_report.json")
        }
    )

    return Command(
        goto="supervisor",
        update={
            "current_dataset_path": output_path,
            "team_reports": append_report(state, report),
            "issue_registry": append_issues(state, issues),
            "messages": [HumanMessage(content="Schema Validation finished.", name="Schema Validation")]
        }
    )


def completeness_analysis_team(state: DataQualityState) -> Command[Literal["supervisor"]]:
    input_path = state["current_dataset_path"]
    df = load_df(input_path)

    tasks_executed = [
        "Detected null and placeholder values",
        "Standardized missing-value markers",
        "Flagged sparse columns",
    ]
    issues_found = []
    changes_applied = []
    changes_not_applied = []
    issues = []

    before_missing = df.isna().sum().sum()
    df = standardize_missing_markers(df)
    after_missing = df.isna().sum().sum()

    if after_missing > before_missing:
        changes_applied.append("Standardized placeholder values to real nulls")

    sparse_cols = []
    for col in df.columns:
        completeness_rate = 1 - df[col].isna().mean()
        if completeness_rate < 0.8:
            issues_found.append(
                f"Low completeness in column '{col}': {completeness_rate:.2%}"
            )
            issues.append(
                {
                    "team": "Completeness Analysis",
                    "issue": f"Low completeness in column '{col}'",
                    "severity": "medium" if completeness_rate >= 0.5 else "high",
                }
            )
        if completeness_rate < 0.5:
            sparse_cols.append(col)

    if sparse_cols:
        changes_not_applied.append(
            f"Sparse columns not dropped automatically: {', '.join(sparse_cols)}"
        )

    output_path = str(ARTIFACT_DIR / "completeness_analysis_output.csv")
    save_df(df, output_path)

    report = build_team_report(
        team_name="Completeness Analysis",
        input_dataset=input_path,
        output_dataset=output_path,
        tasks_executed=tasks_executed,
        issues_found=issues_found if issues_found else ["No completeness issues found"],
        changes_applied=changes_applied if changes_applied else ["No completeness cleaning needed"],
        changes_not_applied=changes_not_applied if changes_not_applied else ["No deferred completeness actions"],
        handoff_summary="Missing markers standardized. Dataset passed to Consistency Validation."
    )

    write_team_report_tool.invoke(
        {
            "report": report,
            "output_path": str(ARTIFACT_DIR / "completeness_analysis_report.json")
        }
    )

    return Command(
        goto="supervisor",
        update={
            "current_dataset_path": output_path,
            "team_reports": append_report(state, report),
            "issue_registry": append_issues(state, issues),
            "messages": [HumanMessage(content="Completeness Analysis finished.", name="Completeness Analysis")]
        }
    )


def consistency_validation_team(state: DataQualityState) -> Command[Literal["supervisor"]]:
    input_path = state["current_dataset_path"]
    df = load_df(input_path)

    tasks_executed = [
        "Trimmed whitespace in text columns",
        "Normalized repeated spaces",
        "Removed exact duplicate rows",
    ]
    issues_found = []
    changes_applied = []
    changes_not_applied = []
    issues = []

    # trim / normalize text columns
    text_cols = df.select_dtypes(include="object").columns.tolist()
    for col in text_cols:
        before = df[col].copy()
        df[col] = df[col].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
        if not before.equals(df[col]):
            changes_applied.append(f"Normalized text formatting in column: {col}")
            issues_found.append(f"Formatting inconsistency found in column: {col}")
            issues.append(
                {
                    "team": "Consistency Validation",
                    "issue": f"Text formatting inconsistency in '{col}'",
                    "severity": "low",
                }
            )

    # remove exact duplicates
    before_rows = len(df)
    df = df.drop_duplicates()
    removed = before_rows - len(df)
    if removed > 0:
        changes_applied.append(f"Removed {removed} exact duplicate rows")
        issues_found.append(f"Detected {removed} exact duplicate rows")
        issues.append(
            {
                "team": "Consistency Validation",
                "issue": f"Exact duplicate rows detected: {removed}",
                "severity": "high",
            }
        )

    output_path = str(ARTIFACT_DIR / "consistency_validation_output.csv")
    save_df(df, output_path)

    report = build_team_report(
        team_name="Consistency Validation",
        input_dataset=input_path,
        output_dataset=output_path,
        tasks_executed=tasks_executed,
        issues_found=issues_found if issues_found else ["No consistency issues found"],
        changes_applied=changes_applied if changes_applied else ["No changes applied"],
        changes_not_applied=changes_not_applied if changes_not_applied else ["No deferred consistency actions"],
        handoff_summary="Formatting normalized and exact duplicates removed where safe. Dataset passed to Anomaly Detection."
    )

    write_team_report_tool.invoke(
        {
            "report": report,
            "output_path": str(ARTIFACT_DIR / "consistency_validation_report.json")
        }
    )

    return Command(
        goto="supervisor",
        update={
            "current_dataset_path": output_path,
            "team_reports": append_report(state, report),
            "issue_registry": append_issues(state, issues),
            "messages": [HumanMessage(content="Consistency Validation finished.", name="Consistency Validation")]
        }
    )


def anomaly_detection_team(state: DataQualityState) -> Command[Literal["supervisor"]]:
    input_path = state["current_dataset_path"]
    df = load_df(input_path)

    tasks_executed = [
        "Detected numeric outliers with IQR method",
        "Detected rare categorical values",
        "Added anomaly flag columns",
    ]
    issues_found = []
    changes_applied = []
    changes_not_applied = []
    issues = []

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    for col in numeric_cols:
        series = df[col].dropna()
        if len(series) < 5:
            continue
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            continue
        low = q1 - 1.5 * iqr
        high = q3 + 1.5 * iqr
        flag_col = f"{col}_outlier_flag"
        df[flag_col] = ((df[col] < low) | (df[col] > high)).fillna(False)
        outlier_count = int(df[flag_col].sum())
        if outlier_count > 0:
            issues_found.append(f"Outliers detected in '{col}': {outlier_count}")
            changes_applied.append(f"Added anomaly flag column: {flag_col}")
            issues.append(
                {
                    "team": "Anomaly Detection",
                    "issue": f"Outliers detected in '{col}': {outlier_count}",
                    "severity": "medium" if outlier_count < 20 else "high",
                }
            )

    object_cols = df.select_dtypes(include="object").columns.tolist()
    for col in object_cols:
        freq = df[col].astype(str).value_counts(dropna=False)
        rare_values = freq[freq == 1]
        if len(rare_values) > 0:
            issues_found.append(f"Rare values detected in '{col}': {len(rare_values)}")
            issues.append(
                {
                    "team": "Anomaly Detection",
                    "issue": f"Rare categorical values in '{col}': {len(rare_values)}",
                    "severity": "low",
                }
            )

    output_path = str(ARTIFACT_DIR / "anomaly_detection_output.csv")
    save_df(df, output_path)

    report = build_team_report(
        team_name="Anomaly Detection",
        input_dataset=input_path,
        output_dataset=output_path,
        tasks_executed=tasks_executed,
        issues_found=issues_found if issues_found else ["No anomalies detected"],
        changes_applied=changes_applied if changes_applied else ["No anomaly flag columns added"],
        changes_not_applied=["Outliers were flagged, not automatically removed"],
        handoff_summary="Anomalies flagged and dataset passed to Remediation."
    )

    write_team_report_tool.invoke(
        {
            "report": report,
            "output_path": str(ARTIFACT_DIR / "anomaly_detection_report.json")
        }
    )

    return Command(
        goto="supervisor",
        update={
            "current_dataset_path": output_path,
            "team_reports": append_report(state, report),
            "issue_registry": append_issues(state, issues),
            "messages": [HumanMessage(content="Anomaly Detection finished.", name="Anomaly Detection")]
        }
    )


def remediation_team(state: DataQualityState) -> Command[Literal["supervisor"]]:
    input_path = state["current_dataset_path"]
    df = load_df(input_path)

    tasks_executed = [
        "Applied safe remediation actions",
        "Kept anomaly flags for auditability",
        "Prepared final cleaned dataset",
    ]
    issues_found = []
    changes_applied = []
    changes_not_applied = []
    issues = []

    # very safe remediations only
    # 1. remove columns fully empty
    empty_cols = [col for col in df.columns if df[col].isna().all()]
    if empty_cols:
        df = df.drop(columns=empty_cols)
        changes_applied.append(f"Removed fully empty columns: {', '.join(empty_cols)}")

    # 2. optional: fill numeric NaN with median only if missing ratio low
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    for col in numeric_cols:
        missing_rate = df[col].isna().mean()
        if 0 < missing_rate <= 0.1:
            median_value = df[col].median()
            df[col] = df[col].fillna(median_value)
            changes_applied.append(
                f"Filled low-rate missing numeric values in '{col}' with median"
            )
        elif missing_rate > 0.1:
            changes_not_applied.append(
                f"Did not impute '{col}' automatically because missing rate is high ({missing_rate:.2%})"
            )

    output_path = str(ARTIFACT_DIR / "final_cleaned_dataset.csv")
    save_df(df, output_path)

    report = build_team_report(
        team_name="Remediation",
        input_dataset=input_path,
        output_dataset=output_path,
        tasks_executed=tasks_executed,
        issues_found=issues_found if issues_found else ["Used upstream issues for remediation"],
        changes_applied=changes_applied if changes_applied else ["No safe remediation needed"],
        changes_not_applied=changes_not_applied if changes_not_applied else ["No deferred remediation actions"],
        handoff_summary="Final cleaned dataset created and returned to the super supervisor."
    )

    write_team_report_tool.invoke(
        {
            "report": report,
            "output_path": str(ARTIFACT_DIR / "remediation_report.json")
        }
    )

    return Command(
        goto="supervisor",
        update={
            "current_dataset_path": output_path,
            "final_dataset_path": output_path,
            "team_reports": append_report(state, report),
            "messages": [HumanMessage(content="Remediation finished.", name="Remediation")]
        }
    )



In [11]:
# ============================================================
# 7. RELIABILITY SCORE + FINAL REPORT
# ============================================================

def compute_reliability_score(issue_registry: List[dict]) -> float:
    score = 100.0
    penalty = {"low": 1.0, "medium": 3.0, "high": 6.0}
    for issue in issue_registry:
        score -= penalty.get(issue.get("severity", "low"), 1.0)
    return max(score, 0.0)


def build_final_report(state: DataQualityState) -> str:
    score = compute_reliability_score(state.get("issue_registry", []))

    lines = []
    lines.append("# FINAL DATA QUALITY REPORT")
    lines.append("")
    lines.append(f"Original dataset: {state['dataset_path']}")
    lines.append(f"Final cleaned dataset: {state.get('final_dataset_path', state['current_dataset_path'])}")
    lines.append(f"Reliability score: {score:.2f}/100")
    lines.append("")

    lines.append("## TEAM AUDIT TRAIL")
    lines.append("")
    for i, report in enumerate(state.get("team_reports", []), start=1):
        lines.append(f"### {i}. {report['team_name']}")
        lines.append(f"- Input dataset: {report['input_dataset']}")
        lines.append(f"- Output dataset: {report['output_dataset']}")
        lines.append("- Tasks executed:")
        for t in report["tasks_executed"]:
            lines.append(f"  - {t}")
        lines.append("- Issues found:")
        for t in report["issues_found"]:
            lines.append(f"  - {t}")
        lines.append("- Changes applied:")
        for t in report["changes_applied"]:
            lines.append(f"  - {t}")
        lines.append("- Changes not applied:")
        for t in report["changes_not_applied"]:
            lines.append(f"  - {t}")
        lines.append(f"- Handoff summary: {report['handoff_summary']}")
        lines.append("")

    lines.append("## GLOBAL SUMMARY")
    lines.append(f"- Total issues tracked: {len(state.get('issue_registry', []))}")
    lines.append(f"- Total teams executed: {len(state.get('team_reports', []))}")
    lines.append("")

    return "\n".join(lines), score


In [12]:
# ============================================================
# 8. DETERMINISTIC SUPER SUPERVISOR
# ============================================================

ORDER = [
    "Schema Validation",
    "Completeness Analysis",
    "Consistency Validation",
    "Anomaly Detection",
    "Remediation",
]


def super_supervisor_node(state: DataQualityState) -> Command[
    Literal["Schema Validation", "Completeness Analysis", "Consistency Validation", "Anomaly Detection", "Remediation", "__end__"]
]:
    completed = [r["team_name"] for r in state.get("team_reports", [])]

    for team_name in ORDER:
        if team_name not in completed:
            return Command(goto=team_name)

    final_report, score = build_final_report(state)

    write_final_report_tool.invoke(
        {
            "final_report": final_report,
            "output_path": str(ARTIFACT_DIR / "final_report.txt")
        }
    )

    return Command(
        goto=END,
        update={
            "final_report": final_report,
            "reliability_score": score,
        }
    )

In [13]:
# ============================================================
# 9. BUILD GRAPH
# ============================================================

builder = StateGraph(DataQualityState)

builder.add_node("supervisor", super_supervisor_node)
builder.add_node("Schema Validation", schema_validation_team)
builder.add_node("Completeness Analysis", completeness_analysis_team)
builder.add_node("Consistency Validation", consistency_validation_team)
builder.add_node("Anomaly Detection", anomaly_detection_team)
builder.add_node("Remediation", remediation_team)

builder.add_edge(START, "supervisor")
builder.add_edge("Schema Validation", "supervisor")
builder.add_edge("Completeness Analysis", "supervisor")
builder.add_edge("Consistency Validation", "supervisor")
builder.add_edge("Anomaly Detection", "supervisor")
builder.add_edge("Remediation", "supervisor")

data_quality_graph = builder.compile()


In [14]:
# ============================================================
# 10. INITIALIZE + RUN
# ============================================================

def initialize_state(dataset_path: str) -> dict:
    working_copy_path = str(ARTIFACT_DIR / f"working_{Path(dataset_path).name}")

    copy_dataset_tool.invoke(
        {
            "dataset_path": dataset_path,
            "output_path": working_copy_path,
        }
    )

    return {
        "messages": [HumanMessage(content=f"Run data quality pipeline on {dataset_path}", name="user")],
        "dataset_path": dataset_path,
        "current_dataset_path": working_copy_path,
        "final_dataset_path": "",
        "team_reports": [],
        "issue_registry": [],
        "reliability_score": 0.0,
        "final_report": "",
    }


if __name__ == "__main__":
    state = initialize_state("attivazioniCessazioni.csv")
    result = data_quality_graph.invoke(state)

    print("\n================ FINAL REPORT ================\n")
    print(result["final_report"])
    print("\nFinal cleaned CSV:", result["final_dataset_path"])


================ FINAL REPORT ================

# FINAL DATA QUALITY REPORT

Original dataset: attivazioniCessazioni.csv
Final cleaned dataset: artifacts/final_cleaned_dataset.csv
Reliability score: 0.00/100

## TEAM AUDIT TRAIL

### 1. Schema Validation
- Input dataset: artifacts/working_attivazioniCessazioni.csv
- Output dataset: artifacts/schema_validation_output.csv
- Tasks executed:
  - Checked column naming conventions
  - Standardized invalid column names
  - Applied safe numeric coercion on clearly numeric columns
- Issues found:
  - Invalid or inconsistent column name: _id
  - Invalid or inconsistent column name: RATA
  - Invalid or inconsistent column name: aggregation-time
  - Invalid or inconsistent column name: Provincia Sede
  - Invalid or inconsistent column name: CODICE ENTE
  - Invalid or inconsistent column name: 3descrizione
  - Invalid or inconsistent column name: regione%sede
  - Invalid or inconsistent column name: att ivazioni
- Changes applied:
  - Renamed 8 co